In [24]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/train.csv")

y = np.log1p(train["log_hours"])
X = train.drop(columns=["hours_spent", "log_hours", "hours_bucket", "project_id"])

d:\Xin Bing\VSC\hour-detective\.venv\Lib\site-packages\pandas\core\arraylike.py:438: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [25]:
def add_features(df):
    df = df.copy()

    df["biggest_commit_share"] = df["max_lines_in_one_commit"] / df["lines_added_total"].replace(0, np.nan)
    df["lines_per_active_day"] = df["lines_of_code"] / df["active_commit_days"].replace(0, np.nan)
    df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"].replace(0, np.nan)
    df["active_day_ratio"] = df["active_commit_days"] / (df["project_span_days"] + 1)

    return df

In [26]:
ORDER_MAPS = {
    "author_prior_projects": {"0": 0, "1-2": 1, "3-5": 2, "6+": 3},
    "author_account_age_years": {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3},
}

def encode_ordered(df):
    df = df.copy()
    for col, mapping in ORDER_MAPS.items():
        df[col] = df[col].astype(str).map(mapping)
    return df

In [27]:
print(train["author_account_age_years"].unique())

<StringArray>
['1-2', '3-5', '<1', '6+']
Length: 4, dtype: str


In [28]:
def encode_categories(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("category")
    return df

In [29]:
def prepare(df):
    df = add_features(df)
    df = encode_ordered(df)
    df = encode_categories(df)
    return df

X = prepare(X)

print(X.shape)
print(X.dtypes.value_counts())
X.head()

(4500, 73)
int64       32
float64     19
bool        12
category     5
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_15956\3138021966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,median_lines_per_commit,...,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,biggest_commit_share,lines_per_active_day,commits_per_active_day,active_day_ratio
0,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,128.5,...,0,0,0,False,0,1,0.188235,306.666667,2.933333,0.263158
1,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,32.0,...,0,0,0,False,0,2,0.800000,100.000000,3.000000,0.038961
2,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,64.5,...,0,0,0,False,0,2,0.525424,1300.000000,12.000000,1.000000
3,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,1904.0,...,0,0,0,False,0,0,0.366667,300.000000,4.000000,1.000000
4,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,198.0,...,0,0,0,False,0,1,0.250000,200.000000,2.333333,0.046154


In [30]:
import lightgbm as lgb
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [31]:
oof = np.zeros(len(X))
models = []

In [ ]:
for fold,(train_idx, test_idx) in enumerate(kf.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    model = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te,y_te)],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    oof[test_idx] = model.predict(X_te)
    models.append(model)
    print(f"Fold {fold}: used {model.best_iteration_} trees")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2932851695.py, line 9)